# Chapter 21: Using Custom Functions, Lambda Expressions, Apply, and Writing Reusable Logic**Companion notebook** for *Beginner's Guide to Pandas* by Ravi ShankarRun each cell in order. Exercises are at the end.

In [ ]:
import pandas as pdimport numpy as np

# Using Custom Functions, Lambda Expressions, Apply, and Writing Reusable Logic## IntroductionOne of pandas' greatest strengths is its flexibility in applying custom logic to your data. Whether you're transforming values, filtering conditionally, or performing complex calculations across rows and columns, understanding how to write and apply custom functions is essential for data manipulation tasks.This chapter explores how to leverage custom functions, lambda expressions, and pandas' powerful `apply()` method to create reusable, efficient data processing pipelines. We'll start with the fundamentals of `apply()` itself, then build toward more advanced patterns.---## Understanding the `apply()` MethodThe `apply()` method is the gateway to custom logic in pandas. It allows you to pass a function to operate on your data — whether that's a Series, DataFrame, or grouped data.### Basic Series ApplicationAt its simplest, `apply()` takes a function and applies it to each element of a Series:

In [ ]:
import pandas as pdimport numpy as np# Create a sample Seriestemperatures = pd.Series([20, 25, 18, 22], index=['Mon', 'Tue', 'Wed', 'Thu'])# Apply a simple transformationcelsius_to_fahrenheit = lambda x: (x * 9/5) + 32temperatures_f = temperatures.apply(celsius_to_fahrenheit)print(temperatures_f)

Output:```Mon    68.0Tue    77.0Wed    64.4Thu    71.6dtype: float64```### DataFrame Application Along AxesWith DataFrames, `apply()` can work along rows or columns using the `axis` parameter:

In [ ]:
sales_data = pd.DataFrame({    'Q1': [100, 150, 200],    'Q2': [120, 160, 210],    'Q3': [130, 170, 220],    'Q4': [140, 180, 230]}, index=['Product_A', 'Product_B', 'Product_C'])# Calculate average across quarters (axis=1 means across columns)avg_sales = sales_data.apply(np.mean, axis=1)print(avg_sales)

Output:```Product_A    122.5Product_B    165.0Product_C    215.0dtype: float64```

In [ ]:
# Find maximum in each quarter (axis=0 means down rows)max_per_quarter = sales_data.apply(np.max, axis=0)print(max_per_quarter)

Output:```Q1    200Q2    210Q3    220Q4    230dtype: int64```**Key Point:** `axis=1` applies the function across each row (producing one result per row), while `axis=0` applies it down each column (producing one result per column).---## Custom Functions with Additional ArgumentsCustom functions often need additional parameters beyond the data itself. The `args` and `kwargs` parameters allow you to pass these values cleanly.### Passing Positional ArgumentsWhen your custom function requires additional parameters, use the `args` parameter to pass them as a tuple:

In [ ]:
def scale_and_shift(x, scale_factor, shift_amount):    """Scale data by a factor, then shift it."""    return (x * scale_factor) + shift_amountmeasurements = pd.Series([10, 20, 30, 40])# Pass additional arguments as a tuplescaled = measurements.apply(scale_and_shift, args=(2, 5))print(scaled)

Output:```0    251    452    653    85dtype: int64```**Key Point:** The `args` parameter accepts a tuple of positional arguments. The first argument to your function is always the data element being processed; the values in `args` follow after it.### Passing Keyword ArgumentsFor more complex scenarios, pass keyword arguments directly to `apply()`:

In [ ]:
def conditional_adjustment(x, **adjustments):    """Apply different adjustments based on conditions."""    result = x    if adjustments.get('apply_discount', False):        result *= (1 - adjustments.get('discount_rate', 0.1))    if adjustments.get('add_tax', False):        result *= (1 + adjustments.get('tax_rate', 0.08))    return resultprices = pd.Series([100, 200, 300])# Apply with keyword argumentsfinal_prices = prices.apply(    conditional_adjustment,    apply_discount=True,    discount_rate=0.15,    add_tax=True,    tax_rate=0.08)print(final_prices)

Output:```0     93.21    186.42    279.6dtype: float64```---## Lambda Expressions: Quick, Inline FunctionsLambda expressions are perfect for simple, one-off transformations without defining a full function. They are anonymous functions defined with minimal syntax.### Basic Lambda Usage

In [ ]:
# Square each temperature valuetemperatures = pd.Series([20, 21, 12], index=['London', 'New York', 'Helsinki'])squared = temperatures.apply(lambda x: x ** 2)print(squared)

Output:```London     400New York   441Helsinki   144dtype: int64```### Conditional Logic with Lambda

In [ ]:
# Categorize temperaturescategories = temperatures.apply(    lambda x: 'Cold' if x < 15 else 'Warm')print(categories)

Output:```London      WarmNew York    WarmHelsinki    Colddtype: object```### Lambda with External Variables

In [ ]:
threshold = 15# Zero out values at or below thresholdfiltered = temperatures.apply(    lambda x: x if x > threshold else 0)print(filtered)

Output:```London     20New York   21Helsinki    0dtype: int64```### Lambda with Multiple ColumnsWhen working with DataFrames, you can use a lambda to access multiple columns in a row:

In [ ]:
transactions = pd.DataFrame({    'quantity': [5, 3, 8, 2],    'unit_price': [10, 25, 15, 50],    'discount_percent': [0, 10, 5, 20]})# Calculate final price with discounttransactions['final_total'] = transactions.apply(    lambda row: row['quantity'] * row['unit_price'] * (1 - row['discount_percent'] / 100),    axis=1)print(transactions)

Output:```   quantity  unit_price  discount_percent  final_total0         5          10                 0         50.01         3          25                10         67.52         8          15                 5        114.03         2          50                20         80.0```---## Apply on Series### Element-wise TransformationThe `apply()` method on a Series processes each element individually:

In [ ]:
prices = pd.Series([100, 250, 75, 150])# Apply a discount functiondef apply_discount(price, discount_percent=10):    return price * (1 - discount_percent / 100)discounted = prices.apply(apply_discount, discount_percent=15)print(discounted)

Output:```0     85.001    212.502     63.753    127.50dtype: float64```### Chaining Multiple Operations

In [ ]:
# Chain multiple transformationsresult = (prices    .apply(lambda x: x * 1.1)       # Add 10% markup    .apply(lambda x: round(x, 2))   # Round to 2 decimals)print(result)

Output:```0    110.01    275.02     82.53    165.0dtype: float64```---## Apply on DataFrames### Row-wise ApplicationApply a function to each row by setting `axis=1`:

In [ ]:
sales_data = pd.DataFrame({    'Product_A': [100, 150, 200],    'Product_B': [80, 120, 180],    'Product_C': [60, 90, 140]})# Calculate total sales per rowdef row_total(row):    return row.sum()totals = sales_data.apply(row_total, axis=1)print(totals)

Output:```0    2401    3602    520dtype: int64```### Column-wise ApplicationApply a function to each column by setting `axis=0` (the default):

In [ ]:
# Calculate average sales per productdef column_average(col):    return col.mean()averages = sales_data.apply(column_average, axis=0)print(averages)

Output:```Product_A    150.000000Product_B    126.666667Product_C     96.666667dtype: float64```### Complex Row-wise Transformations

In [ ]:
# Add a new column with conditional logic based on row valuesdef classify_performance(row):    total = row.sum()    if total > 500:        return 'High'    elif total > 300:        return 'Medium'    else:        return 'Low'sales_data['Performance'] = sales_data.apply(classify_performance, axis=1)print(sales_data)

Output:```   Product_A  Product_B  Product_C Performance0        100         80         60         Low1        150        120         90      Medium2        200        180        140        High```---## Using NumPy Functions with ApplyPandas integrates seamlessly with NumPy functions, which can be passed directly to `apply()`:

In [ ]:
values = pd.Series([1, 10, 100, 1000])# Apply logarithmlog_values = values.apply(np.log10)print(log_values)

Output:```0    0.01    1.02    2.03    3.0dtype: float64```---## Conditional Selection with Where and Mask### Using Where for Conditional ReplacementThe `where()` method keeps values where a condition is True and replaces others with a specified value:

In [ ]:
data = pd.Series([5, -3, 8, -1, 0])# Keep only positive values, replace others with 0positive_only = data.where(data > 0, 0)print(positive_only)

Output:```0    51    02    83    04    0dtype: int64```### Where with Lambda Functions

In [ ]:
df = pd.DataFrame({    'A': [1, 2, 3, 4],    'B': [5, 6, 7, 8]})# Keep values > 3, replace others with -1result = df.where(lambda x: x > 3, -1)print(result)

Output:```   A  B0 -1  51 -1  62 -1  73  4  8```### Using Mask (Inverse of Where)The `mask()` method replaces values where a condition is True — the inverse of `where()`:

In [ ]:
data = pd.Series([5, -3, 8, -1, 0])# Replace negative values with NaNmasked = data.mask(data < 0)print(masked)

Output:```0    5.01    NaN2    8.03    NaN4    0.0dtype: float64```---## Writing Reusable Logic PatternsNamed functions are better than lambdas for complex logic because they are easier to read, test, and reuse across your codebase.### Pattern 1: Validation Functions

In [ ]:
def validate_email(email):    """Check if email format is valid."""    return '@' in email and '.' in email.split('@')[1]emails = pd.Series([    'user@example.com',    'invalid.email',    'admin@company.co.uk'])valid = emails.apply(validate_email)print(valid)

Output:```0     True1    False2     Truedtype: bool```### Pattern 2: Transformation Pipelines

In [ ]:
def clean_and_transform(text):    """Reusable text cleaning function."""    return (text        .strip()           # Remove whitespace        .lower()           # Convert to lowercase        .replace(' ', '_') # Replace spaces with underscores    )names = pd.Series(['  John Doe  ', 'JANE SMITH', 'bob jones '])cleaned = names.apply(clean_and_transform)print(cleaned)

Output:```0      john_doe1    jane_smith2     bob_jonesdtype: object```### Pattern 3: Multi-step Calculations

In [ ]:
def calculate_grade(score):    """Convert numeric score to letter grade."""    if score >= 90:        return 'A'    elif score >= 80:        return 'B'    elif score >= 70:        return 'C'    elif score >= 60:        return 'D'    else:        return 'F'scores = pd.Series([95, 87, 72, 58, 91])grades = scores.apply(calculate_grade)print(grades)

Output:```0    A1    B2    C3    F4    Adtype: object```### Structuring Functions for Full Data PipelinesFor larger workflows, you can chain multiple named functions together into a pipeline. Here is an example that validates, cleans, and categorizes email addresses:

In [ ]:
def validate_and_clean_email(email):    """Validate and standardize email format."""    if not isinstance(email, str):        return None    email = email.strip().lower()    if '@' not in email:        return None    return emaildef extract_email_domain(email):    """Extract domain from a cleaned email."""    if email is None:        return None    return email.split('@')[1]def categorize_domain(domain):    """Categorize domain type."""    if domain is None:        return 'invalid'    if domain.endswith('.edu'):        return 'educational'    elif domain.endswith('.gov'):        return 'government'    elif domain.endswith('.org'):        return 'nonprofit'    else:        return 'commercial'# Apply each step of the pipelineraw_emails = pd.Series([    'JOHN@COMPANY.COM',    'jane@university.edu',    'invalid',    'bob@agency.gov'])cleaned = raw_emails.apply(validate_and_clean_email)domains = cleaned.apply(extract_email_domain)categories = domains.apply(categorize_domain)email_analysis = pd.DataFrame({    'original': raw_emails,    'cleaned': cleaned,    'domain': domains,    'category': categories})print(email_analysis)

Output:```              original              cleaned           domain     category0     JOHN@COMPANY.COM     john@company.com      company.com   commercial1  jane@university.edu  jane@university.edu   university.edu  educational2              invalid                 None             None      invalid3       bob@agency.gov       bob@agency.gov       agency.gov   government```---## Practical Example: Building a Data Cleaning PipelineHere is a more complete example that applies multiple cleaning functions to a customer dataset:

In [ ]:
def clean_phone_number(phone):    """Remove non-digit characters and format as (XXX) XXX-XXXX."""    if pd.isna(phone):        return None    digits = ''.join(c for c in str(phone) if c.isdigit())    if len(digits) == 10:        return f"({digits[:3]}) {digits[3:6]}-{digits[6:]}"    return Nonedef validate_age(age):    """Ensure age is within reasonable bounds."""    try:        age = int(age)        return age if 0 <= age <= 120 else None    except (ValueError, TypeError):        return Nonedef standardize_name(name):    """Capitalize each word in a name properly."""    if pd.isna(name):        return None    return ' '.join(word.capitalize() for word in str(name).split())# Apply the cleaning pipelinecustomer_data = pd.DataFrame({    'name': ['john smith', 'JANE DOE', 'bob johnson'],    'phone': ['555-123-4567', '(555) 987-6543', '5551234567'],    'age': [28, '35', 42]})customer_data['name_clean'] = customer_data['name'].apply(standardize_name)customer_data['phone_clean'] = customer_data['phone'].apply(clean_phone_number)customer_data['age_clean'] = customer_data['age'].apply(validate_age)print(customer_data[['name_clean', 'phone_clean', 'age_clean']])

Output:```     name_clean      phone_clean  age_clean0    John Smith  (555) 123-4567         281      Jane Doe  (555) 987-6543         352  Bob Johnson  (555) 123-4567         42```---## Advanced: Apply with GroupBy OperationsThe `apply()` method becomes even more powerful when combined with `groupby()`, enabling sophisticated group-wise transformations.### Group-wise Transformations

In [ ]:
sales_by_region = pd.DataFrame({    'region': ['North', 'South', 'North', 'South', 'East', 'East'],    'sales': [100, 150, 120, 130, 200, 210],    'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Jan', 'Feb']})# Calculate each sale as a percentage of its regional totaldef percent_of_group(group):    """Calculate each value as percentage of group total."""    group = group.copy()    group['pct_of_region'] = (group['sales'] / group['sales'].sum()) * 100    return groupresult = sales_by_region.groupby('region').apply(percent_of_group)print(result)

Output:```  region  sales month  pct_of_region0  North    100   Jan          45.452  North    120   Feb          54.551  South    150   Jan          53.573  South    130   Feb          46.434   East    200   Jan          48.785   East    210   Feb          51.22```### Custom Aggregations with GroupBy

In [ ]:
def calculate_stats(group):    """Calculate multiple statistics for a group."""    return pd.Series({        'count': len(group),        'mean': group['sales'].mean(),        'std': group['sales'].std(),        'range': group['sales'].max() - group['sales'].min()    })stats = sales_by_region.groupby('region').apply(calculate_stats)print(stats)

Output:```        count   mean        std  rangeregionEast      2.0  205.0   7.071068   10.0North     2.0  110.0  14.142136   20.0South     2.0  140.0  14.142136   20.0```---## Performance Considerations### Apply vs. Vectorized OperationsWhile `apply()` is flexible, vectorized operations are significantly faster because they operate on entire arrays at once rather than looping element by element:

In [ ]:
import time# Create a large serieslarge_series = pd.Series(range(1_000_000))# Method 1: Using apply (slower)start = time.time()result_apply = large_series.apply(lambda x: x * 2 + 5)apply_time = time.time() - start# Method 2: Vectorized operation (faster)start = time.time()result_vectorized = large_series * 2 + 5vectorized_time = time.time() - startprint(f"Apply time:       {apply_time:.4f}s")print(f"Vectorized time:  {vectorized_time:.4f}s")print(f"Speedup: {apply_time / vectorized_time:.1f}x faster with vectorization")

**Recommendation:** Use vectorized operations whenever possible. Reserve `apply()` for complex logic — such as multi-column row calculations, conditional branching, or string parsing — that cannot be easily expressed as a vectorized operation.---## Summary- **`apply()` is your gateway to custom logic** in pandas, working with Series, DataFrames, and grouped data- **Lambda functions** are ideal for simple, inline transformations- **Named functions** are better for complex logic and reusability- **Use `args` and `kwargs`** to pass additional parameters to your functions- **`where()` and `mask()`** enable conditional value replacement without writing explicit loops- **Vectorized operations** are faster than `apply()` — use them when possible- **GroupBy + apply** enables sophisticated group-wise transformations- **Build pipelines** by chaining multiple custom functions together for clean, maintainable data processing

---# ExercisesTest your understanding of this chapter's concepts.

### Exercise 1: Apply a Custom Function to a SeriesCreate a DataFrame of student test scores and write a custom function that converts numeric scores to letter grades. Use apply() to add a new 'grade' column based on the scores.

In [ ]:
import pandas as pd# Sample datadf = pd.DataFrame({    'student': ['Alice', 'Bob', 'Carol', 'David', 'Eva'],    'score': [92, 75, 88, 61, 95]})# TODO: Write a function called 'assign_grade' that takes a score and returns:# 'A' for 90+, 'B' for 80-89, 'C' for 70-79, 'D' for 60-69, 'F' for below 60def assign_grade(score):    pass# TODO: Use apply() on the 'score' column to create a new 'grade' columndf['grade'] = Noneprint(df)

### Exercise 2: Lambda Expressions for Data TransformationGiven a DataFrame of product prices, use lambda expressions with apply() to apply a discount, format prices as strings, and classify products by price range. Practice writing concise inline transformations.

In [ ]:
import pandas as pd# Sample datadf = pd.DataFrame({    'product': ['Widget', 'Gadget', 'Doohickey', 'Thingamajig', 'Whatsit'],    'price': [15.99, 84.50, 5.25, 120.00, 42.75]})# TODO: Use a lambda with apply() to create a 'discounted_price' column# Apply a 10% discount (multiply price by 0.90), rounded to 2 decimal placesdf['discounted_price'] = None# TODO: Use a lambda with apply() to create a 'price_label' column# Label prices as 'budget' if under 20, 'mid-range' if 20-99, 'premium' if 100+df['price_label'] = None# TODO: Use a lambda with apply() to create a 'display_price' column# Format the original price as a string like '$15.99'df['display_price'] = Noneprint(df)

### Exercise 3: Apply with Custom Arguments and DataFrame RowsCreate a DataFrame of employee records and write a custom function that accepts additional arguments. Use apply() with axis=1 to process entire rows and compute a performance bonus based on salary and a configurable bonus rate.

In [ ]:
import pandas as pd# Sample datadf = pd.DataFrame({    'employee': ['Sam', 'Jordan', 'Taylor', 'Morgan', 'Casey'],    'department': ['Sales', 'Engineering', 'Sales', 'HR', 'Engineering'],    'salary': [55000, 95000, 62000, 48000, 105000],    'performance_rating': [4, 5, 3, 4, 5]})# TODO: Write a function called 'calculate_bonus' that takes a row (Series),# and a 'base_rate' argument (a decimal like 0.05).# The bonus = salary * base_rate * (performance_rating / 5)# Return the bonus rounded to 2 decimal places.def calculate_bonus(row, base_rate):    pass# TODO: Use apply() with axis=1 to create a 'bonus' column# Use a base_rate of 0.10 (10%)df['bonus'] = None# TODO: Use apply() with axis=1 and a base_rate of 0.15 for Engineering only# Hint: filter the DataFrame first, then applydf.loc[df['department'] == 'Engineering', 'bonus'] = Noneprint(df)

### Exercise 4: Building a Reusable Data Cleaning PipelineGiven a messy DataFrame of customer records, build a reusable data cleaning pipeline using custom functions and apply(). Chain multiple cleaning steps including stripping whitespace, standardizing formats, and flagging invalid entries using where() and mask().

In [ ]:
import pandas as pd# Messy sample datadf = pd.DataFrame({    'name': ['  alice ', 'BOB', 'Carol  ', ' david', 'EVA  '],    'email': ['alice@example.com', 'bob@EXAMPLE.COM', 'not-an-email', 'david@example.com', 'eva@example'],    'age': [25, -3, 31, 200, 28]})# TODO: Write a function 'clean_name' that strips whitespace and converts to title casedef clean_name(name):    pass# TODO: Write a function 'clean_email' that lowercases the emaildef clean_email(email):    pass# TODO: Write a function 'is_valid_email' that returns True if '@' and '.' are both# present in the email (after cleaning), False otherwisedef is_valid_email(email):    pass# TODO: Apply clean_name to the 'name' columndf['name'] = None# TODO: Apply clean_email to the 'email' columndf['email'] = None# TODO: Use apply() and is_valid_email to create a boolean 'valid_email' columndf['valid_email'] = None# TODO: Use where() to keep ages only if they are between 0 and 120 (inclusive),# otherwise replace with Nonedf['age'] = Noneprint(df)

---# Solutions*Scroll down only after you've attempted the exercises above.*<br><br><br><br><br><br><br><br><br><br>

### Solution 1: Apply a Custom Function to a Series

In [ ]:
import pandas as pdimport numpy as np# Sample datadf = pd.DataFrame({    'student': ['Alice', 'Bob', 'Carol', 'David', 'Eva'],    'score': [92, 75, 88, 61, 95]})# Custom function to assign letter gradesdef assign_grade(score):    if score >= 90:        return 'A'    elif score >= 80:        return 'B'    elif score >= 70:        return 'C'    elif score >= 60:        return 'D'    else:        return 'F'# Use apply() on the 'score' column to create a new 'grade' columndf['grade'] = df['score'].apply(assign_grade)print(df)

### Solution 2: Lambda Expressions for Data Transformation

In [ ]:
import pandas as pdimport numpy as np# Sample datadf = pd.DataFrame({    'product': ['Widget', 'Gadget', 'Doohickey', 'Thingamajig', 'Whatsit'],    'price': [15.99, 84.50, 5.25, 120.00, 42.75]})# Use a lambda with apply() to create a 'discounted_price' columndf['discounted_price'] = df['price'].apply(lambda x: round(x * 0.90, 2))# Use a lambda with apply() to create a 'price_label' columndf['price_label'] = df['price'].apply(    lambda x: 'budget' if x < 20 else ('premium' if x >= 100 else 'mid-range'))# Use a lambda with apply() to create a 'display_price' columndf['display_price'] = df['price'].apply(lambda x: f'${x:.2f}')print(df)

### Solution 3: Apply with Custom Arguments and DataFrame Rows

In [ ]:
import pandas as pdimport numpy as np# Sample datadf = pd.DataFrame({    'employee': ['Sam', 'Jordan', 'Taylor', 'Morgan', 'Casey'],    'department': ['Sales', 'Engineering', 'Sales', 'HR', 'Engineering'],    'salary': [55000, 95000, 62000, 48000, 105000],    'performance_rating': [4, 5, 3, 4, 5]})# Function that takes a row and a base_rate argumentdef calculate_bonus(row, base_rate):    bonus = row['salary'] * base_rate * (row['performance_rating'] / 5)    return round(bonus, 2)# Use apply() with axis=1 to create a 'bonus' column with base_rate of 0.10df['bonus'] = df.apply(calculate_bonus, axis=1, base_rate=0.10)# Override Engineering employees with a higher base_rate of 0.15df.loc[df['department'] == 'Engineering', 'bonus'] = (    df[df['department'] == 'Engineering']    .apply(calculate_bonus, axis=1, base_rate=0.15))print(df)

### Solution 4: Building a Reusable Data Cleaning Pipeline

In [ ]:
import pandas as pdimport numpy as np# Messy sample datadf = pd.DataFrame({    'name': ['  alice ', 'BOB', 'Carol  ', ' david', 'EVA  '],    'email': ['alice@example.com', 'bob@EXAMPLE.COM', 'not-an-email', 'david@example.com', 'eva@example'],    'age': [25, -3, 31, 200, 28]})# Function to strip whitespace and convert to title casedef clean_name(name):    return name.strip().title()# Function to lowercase the emaildef clean_email(email):    return email.lower()# Function to check if an email contains '@' and '.'def is_valid_email(email):    return '@' in email and '.' in email# Apply clean_name to the 'name' columndf['name'] = df['name'].apply(clean_name)# Apply clean_email to the 'email' columndf['email'] = df['email'].apply(clean_email)# Use apply() and is_valid_email to create a boolean 'valid_email' columndf['valid_email'] = df['email'].apply(is_valid_email)# Use where() to keep ages only if between 0 and 120, otherwise replace with Nonedf['age'] = df['age'].where((df['age'] >= 0) & (df['age'] <= 120), other=None)print(df)